# CNN-Transformer Quantization & On-Device Profiling

Quantize the best CNN-Transformer models to INT8 and profile on real mobile hardware.
Tests two checkpoints:
- **Tuned**: Hyperband trial 0 (lr=0.000356, warmup=5) — best from Optuna search
- **Baseline**: From architecture experiments — default hyperparameters

### Pipeline:
1. Load checkpoints and export to ONNX (unrolled transformer for AI Hub compatibility)
2. ONNX Runtime INT8 quantization (QDQ format)
3. Evaluate FP32 vs INT8 AUC on Shaoxing
4. Qualcomm AI Hub: quantize, compile, profile on S24 + Redmi Note 10 5G
5. Apple Core ML conversion + benchmarking
6. Summary comparison

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
import onnx
from onnx.external_data_helper import convert_model_from_external_data
import wfdb
import time
import json as _json
import duckdb
from datetime import datetime

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

os.makedirs("../models", exist_ok=True)
os.makedirs("../cache", exist_ok=True)

db = duckdb.connect("../experiment_results.duckdb")
db.execute("""
    CREATE TABLE IF NOT EXISTS catnet_quantization (
        timestamp TIMESTAMP, checkpoint VARCHAR, format VARCHAR,
        size_mb DOUBLE, shaoxing_auc DOUBLE,
        s24_inference_us DOUBLE, redmi_inference_us DOUBLE,
        notes VARCHAR
    )
""")
print("Ready.")

Using device: mps
Ready.


## 1. CNN-Transformer Architecture (Training + Export versions)

In [2]:
# Training version (uses nn.TransformerEncoder)
class CATNet(nn.Module):
    def __init__(self, num_classes=1, d_model=64, nhead=4, num_encoder_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, d_model, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(d_model), nn.ReLU(),
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, 125, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.cnn(x).transpose(1, 2)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.transformer(x).transpose(1, 2)
        return self.head(self.pool(x).squeeze(-1))

# Export version (unrolled transformer — no SequenceEmpty/Loop ONNX ops)
class ManualTransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_feedforward), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model), nn.Dropout(dropout),
        )
    def forward(self, x):
        x2 = self.norm1(x)
        x = x + self.attn(x2, x2, x2, need_weights=False)[0]
        return x + self.ff(self.norm2(x))

class CATNetExport(nn.Module):
    def __init__(self, num_classes=1, d_model=64, nhead=4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, d_model, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm1d(d_model), nn.ReLU(),
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, 125, d_model))
        self.block1 = ManualTransformerBlock(d_model, nhead, d_model * 4)
        self.block2 = ManualTransformerBlock(d_model, nhead, d_model * 4)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.cnn(x).transpose(1, 2)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.block2(self.block1(x))
        return self.head(self.pool(x.transpose(1, 2)).squeeze(-1))

def transfer_weights(src_state_dict, dst_model):
    """Transfer weights from CATNet to CATNetExport."""
    new_sd = dst_model.state_dict()
    transferred = 0
    for old_key, tensor in src_state_dict.items():
        new_key = old_key
        new_key = new_key.replace('transformer.layers.0.', 'block1.')
        new_key = new_key.replace('transformer.layers.1.', 'block2.')
        new_key = new_key.replace('.self_attn.', '.attn.')
        new_key = new_key.replace('.linear1.', '.ff.0.')
        new_key = new_key.replace('.linear2.', '.ff.3.')
        if new_key in new_sd and tensor.shape == new_sd[new_key].shape:
            new_sd[new_key] = tensor
            transferred += 1
    dst_model.load_state_dict(new_sd)
    print(f"  Transferred {transferred}/{len(src_state_dict)} parameters")
    return dst_model

print("Architectures defined.")

Architectures defined.


## 2. Load Shaoxing Evaluation Data

In [3]:
shaoxing_path = '../data/a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0/'
MI_CODE = '164865005'

def load_shaoxing_samples(base_path, lead_idx=0, limit=None):
    records_file = os.path.join(base_path, 'RECORDS')
    with open(records_file, 'r') as f:
        subfolders = [line.strip() for line in f if line.strip()]
    signals, labels = [], []
    for sub in subfolders:
        if limit and len(signals) >= limit: break
        sub_path = os.path.join(base_path, sub)
        if not os.path.exists(sub_path): continue
        for fname in os.listdir(sub_path):
            if not fname.endswith('.hea'): continue
            if limit and len(signals) >= limit: break
            try:
                rec_path = os.path.join(base_path, sub, fname.replace('.hea', ''))
                signal, meta = wfdb.rdsamp(rec_path)
                s = signal[::5, lead_idx].astype(np.float32)
                s = (s - s.mean()) / (s.std() + 1e-8)
                dx_list = []
                for c in meta['comments']:
                    if 'Dx:' in c:
                        dx_list = [x.strip() for x in c.split('Dx:')[1].strip().split(',')]
                        break
                signals.append(s)
                labels.append(1 if MI_CODE in dx_list else 0)
            except: continue
    return signals, labels

def pad_or_truncate(sig, length=1000):
    if len(sig) >= length: return sig[:length]
    return np.pad(sig, (0, length - len(sig)))

print("Loading Shaoxing data...")
signals, labels = load_shaoxing_samples(shaoxing_path)
print(f"Loaded {len(signals)} samples | MI: {sum(labels)} ({sum(labels)/len(labels)*100:.2f}%)")

Loading Shaoxing data...
Loaded 45150 samples | MI: 123 (0.27%)


## 3. Export Both Checkpoints to ONNX

In [4]:
checkpoints = {
    "tuned": "../checkpoints/catnet_tuning_trial0_finetuned.pt",
    "baseline": "../checkpoints/arch_cat-net_finetuned_best.pt",
}

onnx_paths = {}

for name, ckpt_path in checkpoints.items():
    print(f"\n--- {name} ---")
    onnx_fp32 = f"../models/catnet_{name}_fp32.onnx"
    
    if os.path.exists(onnx_fp32):
        print(f"  Already exists: {onnx_fp32}")
    else:
        # Load training checkpoint
        src_model = CATNet()
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        src_model.load_state_dict(ckpt['model_state_dict'])
        src_model.eval()
        
        # Transfer to export version
        export_model = CATNetExport()
        export_model = transfer_weights(ckpt['model_state_dict'], export_model)
        export_model.eval()
        
        # Verify outputs match
        test_input = torch.randn(1, 1, 1000)
        with torch.no_grad():
            out_src = src_model(test_input).item()
            out_exp = export_model(test_input).item()
        print(f"  Output check — src: {out_src:.6f}, export: {out_exp:.6f}, diff: {abs(out_src - out_exp):.8f}")
        
        # Export to ONNX
        torch.onnx.export(export_model, test_input, onnx_fp32,
            input_names=["ecg_lead"], output_names=["mi_logit"], opset_version=18)
        onnx_model = onnx.load(onnx_fp32, load_external_data=True)
        convert_model_from_external_data(onnx_model)
        onnx.save(onnx_model, onnx_fp32)
        if os.path.exists(onnx_fp32 + ".data"): os.remove(onnx_fp32 + ".data")
    
    size = os.path.getsize(onnx_fp32) / (1024 * 1024)
    print(f"  FP32 ONNX: {size:.2f} MB")
    onnx_paths[name] = onnx_fp32


--- tuned ---
  Already exists: ../models/catnet_tuned_fp32.onnx
  FP32 ONNX: 0.63 MB

--- baseline ---
  Already exists: ../models/catnet_baseline_fp32.onnx
  FP32 ONNX: 0.63 MB


## 4. ONNX Runtime INT8 Quantization

In [5]:
class ECGCalibrationReader(CalibrationDataReader):
    def __init__(self, signals, input_name, num_samples=200):
        self.signals = [pad_or_truncate(sig) for sig in signals[:num_samples]]
        self.input_name = input_name
        self.idx = 0
    def get_next(self):
        if self.idx >= len(self.signals): return None
        sig = self.signals[self.idx]
        self.idx += 1
        return {self.input_name: sig.reshape(1, 1, 1000)}

int8_paths = {}

for name, fp32_path in onnx_paths.items():
    print(f"\n--- {name} INT8 ---")
    int8_path = fp32_path.replace("_fp32.onnx", "_int8.onnx")
    
    if os.path.exists(int8_path):
        print(f"  Already exists: {int8_path}")
    else:
        from onnxruntime.quantization import preprocess
        preprocessed = fp32_path.replace("_fp32.onnx", "_preprocessed.onnx")
        preprocess.quant_pre_process(fp32_path, preprocessed)
        
        session = ort.InferenceSession(preprocessed)
        input_name = session.get_inputs()[0].name
        
        calibration_reader = ECGCalibrationReader(signals, input_name, num_samples=200)
        quantize_static(
            model_input=preprocessed,
            model_output=int8_path,
            calibration_data_reader=calibration_reader,
            quant_format=QuantFormat.QDQ,
            weight_type=QuantType.QInt8,
            activation_type=QuantType.QInt8,
        )
        # Clean up preprocessed
        os.remove(preprocessed)
        print(f"  Quantization complete.")
    
    fp32_size = os.path.getsize(fp32_path) / (1024 * 1024)
    int8_size = os.path.getsize(int8_path) / (1024 * 1024)
    print(f"  FP32: {fp32_size:.2f} MB | INT8: {int8_size:.2f} MB | Ratio: {fp32_size/int8_size:.1f}x")
    int8_paths[name] = int8_path


--- tuned INT8 ---
  Already exists: ../models/catnet_tuned_int8.onnx
  FP32: 0.63 MB | INT8: 0.19 MB | Ratio: 3.4x

--- baseline INT8 ---
  Already exists: ../models/catnet_baseline_int8.onnx
  FP32: 0.63 MB | INT8: 0.19 MB | Ratio: 3.4x


## 5. Evaluate FP32 vs INT8 AUC

In [6]:
def evaluate_onnx(session, signals, labels):
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    preds = []
    for sig in signals:
        inp = pad_or_truncate(sig).reshape(1, 1, 1000)
        logit = session.run([output_name], {input_name: inp})[0][0, 0]
        preds.append(1 / (1 + np.exp(-logit)))
    return roc_auc_score(labels, preds)

def benchmark_onnx(session, signals, n=100):
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    for sig in signals[:10]:
        session.run([output_name], {input_name: pad_or_truncate(sig).reshape(1, 1, 1000)})
    latencies = []
    for sig in signals[:n]:
        inp = pad_or_truncate(sig).reshape(1, 1, 1000)
        start = time.perf_counter()
        session.run([output_name], {input_name: inp})
        latencies.append((time.perf_counter() - start) * 1000)
    return np.median(latencies)

results = []

for name in onnx_paths:
    print(f"\n--- {name} ---")
    
    sess_fp32 = ort.InferenceSession(onnx_paths[name])
    sess_int8 = ort.InferenceSession(int8_paths[name])
    
    auc_fp32 = evaluate_onnx(sess_fp32, signals, labels)
    auc_int8 = evaluate_onnx(sess_int8, signals, labels)
    lat_fp32 = benchmark_onnx(sess_fp32, signals)
    lat_int8 = benchmark_onnx(sess_int8, signals)
    
    fp32_size = os.path.getsize(onnx_paths[name]) / (1024 * 1024)
    int8_size = os.path.getsize(int8_paths[name]) / (1024 * 1024)
    
    print(f"  FP32: AUC={auc_fp32:.4f}, {fp32_size:.2f}MB, {lat_fp32:.2f}ms")
    print(f"  INT8: AUC={auc_int8:.4f}, {int8_size:.2f}MB, {lat_int8:.2f}ms")
    print(f"  AUC retention: {auc_int8/auc_fp32*100:.1f}% | Speedup: {lat_fp32/lat_int8:.2f}x")
    
    results.append({'checkpoint': name, 'format': 'FP32', 'size_mb': fp32_size, 'auc': auc_fp32, 'host_latency_ms': lat_fp32})
    results.append({'checkpoint': name, 'format': 'INT8', 'size_mb': int8_size, 'auc': auc_int8, 'host_latency_ms': lat_int8})


--- tuned ---
  FP32: AUC=0.8421, 0.63MB, 0.35ms
  INT8: AUC=0.8412, 0.19MB, 0.36ms
  AUC retention: 99.9% | Speedup: 0.97x

--- baseline ---
  FP32: AUC=0.8551, 0.63MB, 0.35ms
  INT8: AUC=0.8545, 0.19MB, 0.38ms
  AUC retention: 99.9% | Speedup: 0.93x


## 6. Qualcomm AI Hub: Compile & Profile

In [7]:
import qai_hub as hub

client = hub.Client()
print(f"Connected to AI Hub. {len(client.get_devices())} devices available.")

devices_cfg = {
    "s24": {
        "device": hub.Device("Samsung Galaxy S24 (Family)"),
        "label": "Samsung Galaxy S24 (Snapdragon 8 Gen 3)",
        "runtime": "--target_runtime qnn_context_binary",
    },
    "redmi": {
        "device": hub.Device("Xiaomi Redmi Note 10 5G"),
        "label": "Xiaomi Redmi Note 10 5G (Snapdragon 678)",
        "runtime": None,
    },
}

Connected to AI Hub. 81 devices available.


In [8]:
PROFILE_CACHE = "../cache/catnet_quant_profile_results.json"

if os.path.exists(PROFILE_CACHE):
    with open(PROFILE_CACHE) as f:
        profile_results = _json.load(f)
    print("Loaded cached profile results.")
else:
    profile_results = {}

    # Profile FP32 and INT8 for both checkpoints
    all_models = {}
    for name in onnx_paths:
        all_models[f"{name}_fp32"] = onnx_paths[name]
        all_models[f"{name}_int8"] = int8_paths[name]

    for model_key, model_path in all_models.items():
        profile_results[model_key] = {}
        for dev_key, cfg in devices_cfg.items():
            print(f"\n--- {model_key} on {cfg['label']} ---")
            try:
                kwargs = dict(model=model_path, device=cfg["device"], name=f"catnet_q_{model_key}_{dev_key}")
                if cfg["runtime"]: kwargs["options"] = cfg["runtime"]
                
                compile_job = client.submit_compile_job(**kwargs)
                print(f"  Compile submitted...")
                compile_job.wait()
                status = compile_job.get_status()
                if "FAILED" in str(status.code):
                    print(f"  COMPILE FAILED: {status.message[:120]}")
                    profile_results[model_key][dev_key] = {"status": "compile_failed", "error": status.message[:200]}
                    continue
                
                profile_job = client.submit_profile_job(
                    model=compile_job.get_target_model(), device=cfg["device"],
                    name=f"catnet_q_{model_key}_{dev_key}_profile")
                print(f"  Profile submitted...")
                profile_job.wait()
                pstatus = profile_job.get_status()
                if "FAILED" in str(pstatus.code):
                    print(f"  PROFILE FAILED: {pstatus.message[:120]}")
                    profile_results[model_key][dev_key] = {"status": "profile_failed", "error": pstatus.message[:200]}
                    continue
                
                data = profile_job.download_profile()
                summary = data.get("execution_summary", data) if isinstance(data, dict) else {}
                result = {
                    "status": "success",
                    "inference_us": summary.get("estimated_inference_time"),
                    "peak_memory_bytes": summary.get("estimated_inference_peak_memory"),
                }
                profile_results[model_key][dev_key] = result
                print(f"  inference: {result['inference_us']} us, memory: {int(result['peak_memory_bytes'])/(1024*1024):.0f} MB")
            except Exception as e:
                print(f"  ERROR: {e}")
                profile_results[model_key][dev_key] = {"status": "error", "error": str(e)[:200]}

    with open(PROFILE_CACHE, 'w') as f:
        _json.dump(profile_results, f, indent=2)
    print(f"\nSaved to {PROFILE_CACHE}")


--- tuned_fp32 on Samsung Galaxy S24 (Snapdragon 8 Gen 3) ---
Uploading catnet_tuned_fp32.onnx


100%|██████████| 649k/649k [00:00<00:00, 774kB/s]  


Scheduled compile job (jp1dv4vnp) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp1dv4vnp/

  Compile submitted...
Waiting for compile job (jp1dv4vnp) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Scheduled profile job (j57j7y7n5) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j57j7y7n5/

  Profile submitted...
Waiting for profile job (j57j7y7n5) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
  inference: 151 us, memory: 96 MB

--- tuned_fp32 on Xiaomi Redmi Note 10 5G (Snapdragon 678) ---
Uploading catnet_tuned_fp32.onnx


100%|██████████| 649k/649k [00:00<00:00, 817kB/s]  


Scheduled compile job (jpx7dkd8g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jpx7dkd8g/

  Compile submitted...
Waiting for compile job (jpx7dkd8g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Scheduled profile job (jp01vz60g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp01vz60g/

  Profile submitted...
Waiting for profile job (jp01vz60g) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
  inference: 1961 us, memory: 74 MB

--- tuned_int8 on Samsung Galaxy S24 (Snapdragon 8 Gen 3) ---
Uploading catnet_tuned_int8.onnx


100%|██████████| 194k/194k [00:00<00:00, 322kB/s] 


... (7 outputs left out)

100%|██████████| 194k/194k [00:00<00:00, 301kB/s] 


Scheduled compile job (j5q7m6yeg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/j5q7m6yeg/

  Compile submitted...
Waiting for compile job (j5q7m6yeg) completion. Type Ctrl+C to stop waiting at any time.
    ❌ FAILED               Conversion to context binary failed with exit code 14            
  COMPILE FAILED: Conversion to context binary failed with exit code 14

--- baseline_int8 on Xiaomi Redmi Note 10 5G (Snapdragon 678) ---
Uploading catnet_baseline_int8.onnx


100%|██████████| 194k/194k [00:00<00:00, 301kB/s]  


Scheduled compile job (jp34wj9mg) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp34wj9mg/

  Compile submitted...
Waiting for compile job (jp34wj9mg) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
Scheduled profile job (jgo142r15) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jgo142r15/

  Profile submitted...
Waiting for profile job (jgo142r15) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          
  inference: 4158 us, memory: 75 MB

Saved to ../cache/catnet_quant_profile_results.json


## 7. Apple Core ML

In [9]:
import coremltools as ct

coreml_results = {}

for name, ckpt_path in checkpoints.items():
    print(f"\n--- {name} Core ML ---")
    fp32_mlpackage = f"../models/catnet_{name}.mlpackage"
    int8_mlpackage = f"../models/catnet_{name}_int8.mlpackage"
    
    if os.path.exists(fp32_mlpackage) and os.path.exists(int8_mlpackage):
        print("  Loading from disk...")
        ml_fp32 = ct.models.MLModel(fp32_mlpackage)
        ml_int8 = ct.models.MLModel(int8_mlpackage)
    else:
        # Load and export
        src_model = CATNet()
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        src_model.load_state_dict(ckpt['model_state_dict'])
        src_model.eval()
        
        exported = torch.export.export(src_model, (torch.randn(1, 1, 1000),))
        exported = exported.run_decompositions({})
        
        ml_fp32 = ct.convert(exported, minimum_deployment_target=ct.target.macOS13)
        ml_fp32.save(fp32_mlpackage)
        print(f"  FP32 saved to {fp32_mlpackage}")
        
        from coremltools.optimize.coreml import OpLinearQuantizerConfig, OptimizationConfig, linear_quantize_weights
        quant_config = OptimizationConfig(
            global_config=OpLinearQuantizerConfig(mode="linear_symmetric", dtype="int8"))
        ml_int8 = linear_quantize_weights(ml_fp32, config=quant_config)
        ml_int8.save(int8_mlpackage)
        print(f"  INT8 saved to {int8_mlpackage}")
    
    # Benchmark
    coreml_input_name = ml_fp32.input_description._fd_spec[0].name
    
    def bench_coreml(model, n=100):
        for sig in signals[:10]:
            model.predict({coreml_input_name: pad_or_truncate(sig).reshape(1, 1, 1000)})
        lats = []
        for sig in signals[:n]:
            start = time.perf_counter()
            model.predict({coreml_input_name: pad_or_truncate(sig).reshape(1, 1, 1000)})
            lats.append((time.perf_counter() - start) * 1000)
        return np.median(lats)
    
    lat_fp32 = bench_coreml(ml_fp32)
    lat_int8 = bench_coreml(ml_int8)
    print(f"  Core ML FP32: {lat_fp32:.2f} ms | INT8: {lat_int8:.2f} ms | Speedup: {lat_fp32/lat_int8:.2f}x")
    
    coreml_results[name] = {'fp32_ms': lat_fp32, 'int8_ms': lat_int8}

scikit-learn version 1.7.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.10.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.
Skipping import of cpp extensions due to incompatible torch version 2.10.0 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0405 00:34:47.908000 75799 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



--- tuned Core ML ---


/Users/gabrielbutler/.pyenv/versions/3.11.9/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


NotImplementedError: Unsupported fx node alias, kind alias

## 8. Summary

In [ ]:
print("=" * 85)
print("  CNN-Transformer Quantization & Deployment Summary")
print("=" * 85)

# ONNX results
print("\n--- ONNX Runtime (Host) ---")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False, float_format='%.4f'))

# AI Hub results
print("\n--- Qualcomm AI Hub (On-Device) ---")
print(f"{'Model':<25s} {'S24 (us)':>10s} {'S24 Mem':>10s} {'Redmi (us)':>12s} {'Redmi Mem':>10s}")
print("-" * 75)
for model_key in profile_results:
    s24 = profile_results[model_key].get("s24", {})
    redmi = profile_results[model_key].get("redmi", {})
    s24_t = str(s24.get("inference_us", "N/A")) if s24.get("status") == "success" else s24.get("status", "N/A")
    s24_m = f"{s24.get('peak_memory_bytes', 0)/(1024*1024):.0f} MB" if s24.get("status") == "success" else "N/A"
    r_t = str(redmi.get("inference_us", "N/A")) if redmi.get("status") == "success" else redmi.get("status", "N/A")
    r_m = f"{redmi.get('peak_memory_bytes', 0)/(1024*1024):.0f} MB" if redmi.get("status") == "success" else "N/A"
    print(f"{model_key:<25s} {s24_t:>10s} {s24_m:>10s} {r_t:>12s} {r_m:>10s}")

# Core ML results
print("\n--- Apple Core ML (Apple Silicon) ---")
for name, data in coreml_results.items():
    print(f"  {name}: FP32={data['fp32_ms']:.2f}ms, INT8={data['int8_ms']:.2f}ms")

# Log to DuckDB
for r in results:
    s24_data = profile_results.get(f"{r['checkpoint']}_{r['format'].lower()}", {}).get("s24", {})
    redmi_data = profile_results.get(f"{r['checkpoint']}_{r['format'].lower()}", {}).get("redmi", {})
    db.execute("INSERT INTO catnet_quantization VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
        [datetime.now(), r['checkpoint'], r['format'], r['size_mb'], r['auc'],
         s24_data.get('inference_us'), redmi_data.get('inference_us'),
         f"host_latency={r['host_latency_ms']:.2f}ms"])

db.close()
print("\nResults logged to DuckDB.")